In [1]:
import os,sys
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import io

In [2]:
from scipy import sparse

In [3]:
from anndata import AnnData

In [4]:
high_dir = "/home/wergillius/Project/diffuse_differentiate/data/Perturb_sci/"

In [5]:
ontarget_dir = "/home/wergillius/Project/diffuse_differentiate/data/Perturb_sci/ontarget_dir"
meta = pd.read_csv(os.path.join(ontarget_dir, "GSM6752591_on_target_cell_metadata.csv"))
meta = meta.set_index("cell_names")

# transcript

In [32]:
# read mtx, genes, cellbarcodes 
# then create anndata

def read_mtx(directory, prefix):
    """
    read unzipped mtx, genes.tsv , cellbarcode.tsv

    directory : 
    prefix : name before
    """
    matrix = io.mmread(os.path.join(directory, prefix + "matrix.mtx"))

    barcode_df = pd.read_csv(
        os.path.join(directory, prefix + "barcodes.tsv"),
        names = ['cb'])

    gene_df = pd.read_csv(
        os.path.join(directory, prefix + "genes.tsv"), names=['Gene_Symbol'])

    adata = AnnData(
        X = sparse.csr_matrix(matrix.T),
        var = gene_df,
        obs = barcode_df
    )

    return adata

In [20]:
whole_tx_dir = "/home/wergillius/Project/diffuse_differentiate/data/Perturb_sci/tx_dir"
tx_adata = read_mtx(whole_tx_dir, "GSM6752591_on_target_whole_tx.")

/home/wergillius/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/wergillius/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [26]:
tx_adata.X = sparse.csr_matrix(tx_adata.X)

In [6]:
tx_adata = sc.read_h5ad(
    os.path.join(high_dir, "whole_tx_count_Jan13.h5ad" )
)

In [22]:
tx_adata.obs = meta.loc[tx_adata.obs['cb'].tolist()]

In [24]:
tx_adata.write_h5ad(
    os.path.join(high_dir, "whole_tx_count_Jan13.h5ad" )
)

# Nascent RNA

In [21]:
nascent_dir = "/home/wergillius/Project/diffuse_differentiate/data/Perturb_sci/nascent_dir"
nascent_prefix = 'GSM6752591_on_target_nascent_tx.'

In [22]:
nsct_adata = read_mtx(nascent_dir, nascent_prefix)

/home/wergillius/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/wergillius/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [7]:
nsct_adata = sc.read_h5ad(
    os.path.join(high_dir, "nascent_tx_count_Jan13.h5ad" )
)

In [19]:
nsct_adata.obs = meta.loc[nsct_adata.obs['cb'].tolist()]

In [21]:
nsct_adata.write_h5ad(
    os.path.join(high_dir, "nascent_tx_count_Jan13.h5ad" )
)

# on target

In [30]:
ontarget_dir = "/home/wergillius/Project/diffuse_differentiate/data/Perturb_sci/ontarget_dir"
ontarget_prefix = 'GSM6752591_on_target_sgRNA.'

In [33]:
ontarget_adata = read_mtx(ontarget_dir, ontarget_prefix)

/home/wergillius/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/wergillius/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [36]:
ontarget_adata.obs = meta.loc[meta.ontaret]

,cb
0,220824perturbsci_06.GCCGACGATACTTGCGCCGC
1,220824perturbsci_01.CGGACTGGCCTGGTCAGCCA
2,220824perturbsci_07.GTAGTAGTCCACGCGAGATT
3,220824perturbsci_01.AGCTCATTCGGGCAGGTATT
4,220824perturbsci_04.GGCAGCAGTTGCGTTGGAGC
...,...
98310,221010perturbsci_09.GAATCTTAGGACCGCCAACC
98311,221010perturbsci_12.AACTGACCGGGAAGATCGAG
98312,221010perturbsci_13.AAGGTAACCGACTTAACCTT
98313,221010perturbsci_15.CCGATATAAGGCTGGAACTT


In [37]:
ontarget_adata.write_h5ad(
    os.path.join(high_dir, "ontarget_count_Jan12.h5ad")
)

In [8]:
ontarget_adata = sc.read_h5ad(
    os.path.join(high_dir, "ontarget_count_Jan12.h5ad")
)

# save as two different layers

In [16]:
tx_adata.X = tx_adata.X.astype('float64')
tx_adata.layers['Count_whole_tx'] = tx_adata.X.copy().astype("float64")
tx_adata.layers['Count_nascent_tx'] = nsct_adata.X.copy().astype("float64")

In [11]:
tx_adata.obsm['ontarget_sgRNA'] = ontarget_adata.X

In [17]:
tx_adata.write_h5ad(
    os.path.join(high_dir, "multi_modal_count_Jan13.h5ad")
)

In [25]:
mul_adata = sc.read_h5ad(
    os.path.join(high_dir, "multi_modal_count_Jan13.h5ad")
)

In [28]:
mul_adata.obs = tx_adata.obs.loc[mul_adata.obs['cb'].tolist()]

In [29]:
mul_adata

AnnData object with n_obs × n_vars = 98315 × 59429
    obs: 'UMI_counts', 'nascent_UMI_counts', 'nascent_ratio', 'target', 'target_genes', 'gRNA_UMI_counts', 'MT_ratio', 'nascent_MT_ratio', 'Cell_cycle_phase', 'whole_exon_ratio', 'new_exon_ratio'
    var: 'Gene_Symbol'
    layers: 'Count_nascent_tx', 'Count_whole_tx'